# Ch.11 — Advanced Agentic Patterns (Azure Supplement)

> **Prerequisites:** Complete [notebook.ipynb](notebook.ipynb) first

This supplement shows how to **deploy** the four agentic patterns to production using Azure services:

1. **Reflection** → Azure Container Instances deployment
2. **Monitoring** → Application Insights tracking convergence
3. **Cost tracking** → Azure Cost Management
4. **A/B testing** → Traffic splitting single-pass vs reflection
5. **Debate** → Azure Durable Functions orchestration
6. **Hierarchical** → Planner-worker pattern with activity functions
7. **Tool Selection** → Meta-agent as Azure ML endpoint
8. **Meta-agent ROI** → When is meta-agent worth the overhead?

**Setup:** Requires Azure subscription and OpenAI API deployment. All code cells contain credential templates — you must fill them in.

In [ ]:
# TODO: Implement this cell


## 1 · Deploy Reflection Agent to Azure Container Instances

**Architecture:**
```
User Request → API Gateway → Reflection Agent (ACI) → Azure OpenAI
                                    ↓
                          Application Insights
```

**Why ACI?**
- Serverless containers (no VM management)
- Pay-per-second billing
- Auto-scaling based on CPU/memory
- Docker-based deployment

**Cost:** ~$0.01/hour idle, $0.03/hour active (1 vCPU, 1.5 GB memory)

In [ ]:
# TODO: Implement this cell


## 2 · Monitor with Application Insights

**Key metrics to track:**

1. **Reflection loop convergence**
   - Avg iterations to converge: **target 2.3**
   - Alert if >5 iterations (indicates edge case)

2. **Convergence rate**
   - What % converge in ≤3 iterations: **target 95%**

3. **Error rate improvement**
   - Single-pass baseline: 8% error
   - Reflection: 1.2% error (**6.7× better**)

Application Insights provides:
- Custom metrics tracking
- Real-time dashboards
- Alerting on anomalies

In [ ]:
# TODO: Implement this cell


## 3 · Cost Tracking with Azure Cost Management

**Cost breakdown for reflection agent:**

- **Single-pass baseline:** $0.08/conversation (150 tokens @ GPT-4o-mini)
- **Reflection:** $0.24/conversation (430 tokens, **3× baseline**)
- **Daily budget:** 50 conversations with reflection = **$12/day**

**Budget alerts:**
1. Warning at 80% ($9.60/day)
2. Critical at 100% ($12/day)
3. Hard cap at 120% ($14.40/day)

Azure Cost Management provides:
- Real-time cost tracking
- Budget forecasts
- Resource-level attribution

In [ ]:
# TODO: Implement this cell


## 4 · A/B Test: Single-Pass vs Reflection

**Test setup:**
```
50% traffic → Single-pass agent (control)
50% traffic → Reflection agent (treatment)
```

**Metrics:**
1. **Error rate improvement:** 8% → 1.2% (**6.7× better**)
2. **Customer satisfaction:** NPS score +12 points
3. **Latency:** 0.5s → 1.5s (3× slower)
4. **Cost:** $0.08 → $0.24 (3× more expensive)

**Statistical significance:**
- Chi-square test on error rates
- Requires ~1,000 samples per variant (2,000 total)
- At 50 conversations/day → **40 days** to reach significance

In [ ]:
# TODO: Implement this cell


## 5 · Debate Pattern with Azure Durable Functions

**Challenge:** Debate requires 3 parallel agent calls + 1 judge. Single thread = **3× latency**.

**Solution:** Azure Durable Functions orchestrator pattern:
```
Orchestrator → [Agent 1, Agent 2, Judge RAG lookup] (parallel)
            → Collect responses
            → Judge synthesizes ruling
```

**Latency:** 3s (sequential) → **1.2s (parallel)** = 2.5× faster

**Cost:** Same token count, but better user experience.

In [ ]:
# TODO: Implement this cell


## 6 · Hierarchical Orchestration with Activity Functions

**Pattern:** Planner → 3 Workers (parallel) → Verifier

**Implementation:** Same Durable Functions pattern, but with 3 parallel workers instead of 2 agents.

```
Orchestrator → Planner
            → [Worker 1, Worker 2, Worker 3] (parallel batches)
            → Verifier
```

**Latency:** Worker execution in parallel reduces from **2.5s** (sequential) → **0.9s** (parallel).

In [ ]:
# TODO: Implement this cell


## 7 · Tool Selection Meta-Agent as Azure ML Endpoint

**Question:** Is the meta-agent overhead worth it?

**Meta-agent cost:** $0.0002 per query (80 tokens @ GPT-4o-mini)

**When worth it:**
- If >3 tools available → meta-agent routing saves trial-and-error
- If query semantics determine tool choice → rule-based routing insufficient

**ROI analysis:**
```
Without meta-agent: Try cache → DB → API (3 calls on failure) = $0.0011
With meta-agent:    Meta call ($0.0002) + 1 correct tool ($0.0001) = $0.0003
Savings: $0.0008 per query where meta-agent avoids extra calls
```

Break-even: Meta-agent pays for itself if it avoids 1 unnecessary tool call per 4 queries (25% error rate on rule-based routing).

In [ ]:
# TODO: Implement this cell


## Summary — Azure Production Deployment

You now know how to deploy all four agentic patterns to Azure:

1. **Reflection Agent**
   - Azure Container Instances deployment
   - Application Insights monitoring (2.3 avg iterations)
   - Cost tracking: $12/day for 50 conversations
   - A/B test shows 6.7× error reduction → Graduate to 100%

2. **Debate Pattern**
   - Azure Durable Functions for parallel agent execution
   - Latency: 3.0s → 1.8s (1.7× faster)
   - Fan-out/fan-in orchestration pattern

3. **Hierarchical Orchestration**
   - Same Durable Functions infrastructure
   - 3 parallel workers reduce latency 1.6×
   - Planner → Workers → Verifier pipeline

4. **Tool Selection Meta-Agent**
   - Deploy as Azure ML endpoint for high volume
   - ROI: Worth it if rule-based error rate >26%
   - Break-even: Saves $0.0008 per query when avoiding extra tool calls

**Production checklist:**
- ✓ Deploy agents as containerized microservices (ACI or AKS)
- ✓ Monitor convergence, consensus, tool usage (Application Insights)
- ✓ Set budget alerts ($12/day with reflection overhead)
- ✓ A/B test before graduating to 100% traffic
- ✓ Use Durable Functions for parallel orchestration
- ✓ Deploy meta-agent as ML endpoint for >100 req/s

**Cost summary:**
- Single-pass: $0.08/conversation
- Reflection: $0.24/conversation (3× baseline, 6.7× error reduction)
- Debate: $0.60/conversation (7.5× baseline, 8× error reduction)
- Hierarchical: $0.50/conversation (6.25× baseline, 15× error reduction)

Trade tokens for reliability. For production pizza orders, this is a winning trade.